# MRMS Module Demo

This notebook demonstrates how to use the `mrms` module to download and process NOAA MRMS (Multi-Radar/Multi-Sensor System) Composite Reflectivity (CREF) data.

Data is sourced from NOAA's public AWS S3 bucket as hourly GRIB2 files and returned as xarray Datasets in EPSG:4326.

The module exposes three public functions:
- `download_mrms()` — download and decompress a single CREF GRIB2 file
- `process_mrms()` — open a local GRIB2 file, apply bbox, return xarray Dataset
- `get_data()` — high-level wrapper for a full time range, returns a time-concatenated Dataset

In [ ]:
import sys
import os

# if running from the datamods/ directory, this import works directly
# otherwise adjust the path below to point to the datamods/ folder
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import mrms

# directory where downloaded .grib2 files will be stored
DATA_DIR = '/netfiles/ciroh/qpeData'

---
## 1. Download a single file

`download_mrms()` fetches one MRMS CREF GRIB2 file for a given date/time from AWS S3.
If the file is already on disk it will be skipped automatically.

In [ ]:
grib_file = mrms.download_mrms(
    date='20251201_0000',
    data_dir=DATA_DIR
)
print(f'Downloaded to: {grib_file}')

---
## 2. Process a single file

`process_mrms()` opens the GRIB2 file with `cfgrib`, normalizes longitudes to -180–180°,
filters to a bounding box, and returns an xarray Dataset.

By default the bounding box covers the continental United States (CONUS).

In [ ]:
ds = mrms.process_mrms(grib_file)
ds

In [ ]:
print(f'Variable: cref')
print(f'Shape:    {ds["cref"].shape}')
print(f'Lat range: {float(ds.latitude.min()):.3f} to {float(ds.latitude.max()):.3f}')
print(f'Lon range: {float(ds.longitude.min()):.3f} to {float(ds.longitude.max()):.3f}')
print(f'\ncref summary:')
import numpy as np
vals = ds['cref'].values
print(f'  min: {np.nanmin(vals):.2f}')
print(f'  max: {np.nanmax(vals):.2f}')
print(f'  mean: {np.nanmean(vals):.2f}')

### Custom bounding box

Pass any `bbox` dict to crop the grid to a region of interest.

In [ ]:
# example: Ohio River Valley region
orv_bbox = {
    'min_lon': -92.0,
    'max_lon': -78.0,
    'min_lat': 35.0,
    'max_lat': 42.0
}

ds_orv = mrms.process_mrms(grib_file, bbox=orv_bbox)
print(f'Full CONUS grid: {ds["cref"].shape}')
print(f'Ohio River Valley grid: {ds_orv["cref"].shape}')
ds_orv

---
## 3. Quick plot

xarray's built-in `.plot()` method renders the CREF grid directly.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 7))
ds['cref'].plot(
    ax=ax,
    cmap='gist_ncar',
    vmin=0,
    vmax=75,
    cbar_kwargs={'label': 'Composite Reflectivity (dBZ)'}
)
ax.set_title('MRMS CREF — 2025-12-01 00:00 UTC')
plt.tight_layout()
plt.show()

---
## 4. Get data for a time range

`get_data()` downloads all hourly CREF files between `start_datetime` and `end_datetime`
(inclusive) and returns a single xarray Dataset with a `time` dimension.

In [ ]:
ds_range = mrms.get_data(
    start_datetime='20251201_0000',
    end_datetime='20251201_0200',
    data_dir=DATA_DIR
)
ds_range

In [ ]:
print(f'Dimensions: {dict(ds_range.dims)}')
print(f'Timestamps: {ds_range.time.values}')

### Select a single timestep from the time-range result

In [ ]:
import datetime as dt
import numpy as np

ts = np.datetime64(dt.datetime(2025, 12, 1, 1, 0))  # 01:00 UTC
ds_1h = ds_range.sel(time=ts)
print(f'Shape at {ts}: {ds_1h["cref"].shape}')
ds_1h

### Plot all timesteps side-by-side

In [ ]:
n_times = len(ds_range.time)
fig, axes = plt.subplots(1, n_times, figsize=(7 * n_times, 5), constrained_layout=True)

for ax, ts in zip(axes, ds_range.time.values):
    ds_range['cref'].sel(time=ts).plot(
        ax=ax,
        cmap='gist_ncar',
        vmin=0,
        vmax=75,
        add_colorbar=False
    )
    ax.set_title(str(ts)[:16])

fig.colorbar(
    axes[0].collections[0], ax=axes,
    label='Composite Reflectivity (dBZ)', shrink=0.8
)
plt.show()

---
## 5. Combine with a custom bbox over a time range

In [ ]:
southeast_bbox = {
    'min_lon': -92.0,
    'max_lon': -75.0,
    'min_lat': 28.0,
    'max_lat': 38.0
}

ds_se = mrms.get_data(
    start_datetime='20251201_0000',
    end_datetime='20251201_0200',
    bbox=southeast_bbox,
    data_dir=DATA_DIR   # files already downloaded — will be skipped
)

print(f'Southeast region dims: {dict(ds_se.dims)}')
ds_se

---
## 6. Basic xarray operations on the result

Because the output is a standard `xr.Dataset`, you can use the full xarray API.

In [ ]:
# time-mean composite reflectivity across the 3-hour window
mean_cref = ds_range['cref'].mean(dim='time')

fig, ax = plt.subplots(figsize=(14, 7))
mean_cref.plot(ax=ax, cmap='gist_ncar', vmin=0, vmax=75,
               cbar_kwargs={'label': 'Mean CREF (dBZ)'})
ax.set_title('3-Hour Mean MRMS CREF — 2025-12-01 00:00–02:00 UTC')
plt.tight_layout()
plt.show()

In [ ]:
# fraction of the domain with reflectivity > 35 dBZ at each timestep
threshold = 35.0
frac_above = (ds_range['cref'] > threshold).mean(dim=['latitude', 'longitude'])
for ts, frac in zip(ds_range.time.values, frac_above.values):
    print(f'{str(ts)[:16]}  →  {frac * 100:.2f}% of grid > {threshold} dBZ')

---
## Module constants reference

In [ ]:
print('CONUS bounding box:')
print(mrms.CONUS_BBOX)

print('\nProduct:')
print(mrms.MRMS_PRODUCT)

print('\nAWS S3 URL template:')
print(mrms.MRMS_BASE_URL)